In [ ]:
# Initial analysis to pivot the BRK metrics by year, creating separate Parquet files for each year. 
# This allows us to work with smaller datasets in memory and can speed up subsequent analyses that focus on specific years.

import duckdb
from pathlib import Path

# Paths relative to notebooks/
input_file = "../data/raw/brk_metrics.parquet"
output_dir = Path("../data/processed/pivot_by_year")
db_path = "../data/processed/brk_work.duckdb"
temp_dir = "../data/processed/duckdb_temp"

output_dir.mkdir(parents=True, exist_ok=True)
Path(temp_dir).mkdir(parents=True, exist_ok=True)

con = duckdb.connect(db_path)

con.execute("SET threads=1")
con.execute("SET preserve_insertion_order=false")
con.execute("SET memory_limit='5GB'")
con.execute(f"SET temp_directory='{temp_dir}'")

years = con.execute(f"""
SELECT DISTINCT year(day_utc)::INT AS yr
FROM read_parquet('{input_file}')
ORDER BY yr
""").fetchall()

print("Years found:", years)

for (yr,) in years:
    print(f"Processing year {yr}")

    output_file = output_dir / f"brk_pivot_{yr}.parquet"

    con.execute(f"""
    COPY (
        PIVOT (
            SELECT day_utc, metric, value
            FROM read_parquet('{input_file}')
            WHERE year(day_utc) = {yr}
        )
        ON metric
        USING first(value)
        GROUP BY day_utc
    )
    TO '{output_file.as_posix()}'
    (FORMAT PARQUET)
    """)

con.close()